# Initial data quality checks and normalized EDA

This notebook inspects the monthly GitHub language activity dataset and builds interactive Plotly views for exploration. The dataset samples one day per month, so raw counts are useful for validation and hover context, but comparative charts use monthly shares, percentages, ranks, or ratios.

In [37]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.colors import qualitative

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

DATA_PATH = next(
    path
    for path in [
        Path("../data/github_language_activity_monthly.csv"),
        Path("data/github_language_activity_monthly.csv"),
    ]
    if path.exists()
)

df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(dict(year=df["year"], month=df["month"], day=1))

metric_cols = [
    "pushes",
    "pull_requests",
    "issues",
    "issue_comments",
    "stars",
    "forks",
    "creates",
    "contributors",
    "active_repos",
]

event_metric_cols = [
    "pushes",
    "pull_requests",
    "issues",
    "issue_comments",
    "stars",
    "forks",
    "creates",
]

# Set this to False to focus charts on broad programming-language labels.
INCLUDE_ALL_GITHUB_LANGUAGE_LABELS = True
NON_GENERAL_PURPOSE_LABELS = {
    "ASP.NET",
    "Assembly",
    "CSS",
    "Dockerfile",
    "HTML",
    "Jupyter Notebook",
    "Makefile",
    "Markdown",
    "Nix",
    "Procfile",
    "Rich Text Format",
    "Shell",
    "Tex",
    "Vim Script",
    "Vue",
}

def language_scope(data=df, include_all=INCLUDE_ALL_GITHUB_LANGUAGE_LABELS):
    if include_all:
        return data.copy()
    return data.loc[~data["language"].isin(NON_GENERAL_PURPOSE_LABELS)].copy()

LANGUAGE_COLOR_SEQUENCE = list(dict.fromkeys(
    qualitative.Alphabet
    + qualitative.Dark24
    + qualitative.Light24
    + qualitative.Set3
    + qualitative.Pastel
    + qualitative.Bold
))


def language_color_map(languages):
    languages = list(languages)
    if len(languages) > len(LANGUAGE_COLOR_SEQUENCE):
        raise ValueError(
            f"Need {len(languages)} unique colors, but only "
            f"{len(LANGUAGE_COLOR_SEQUENCE)} are configured."
        )
    return {
        language: LANGUAGE_COLOR_SEQUENCE[index]
        for index, language in enumerate(languages)
    }

POPULARITY_SCORE_METRICS = [
    "active_repos",
    "contributors",
    "pull_requests",
    "pushes",
    "stars",
]


def language_popularity_table(data, score_metrics=POPULARITY_SCORE_METRICS):
    totals = data.groupby("language")[score_metrics].sum()
    metric_weights = totals.div(totals.sum(axis=0), axis=1)
    result = totals.copy()
    result["popularity_score"] = metric_weights.mean(axis=1)
    return result.sort_values("popularity_score", ascending=False)


def top_languages_by_popularity(top_n=20, include_all=INCLUDE_ALL_GITHUB_LANGUAGE_LABELS):
    scoped = language_scope(df, include_all=include_all)
    return language_popularity_table(scoped).head(top_n).index.tolist()

analysis_df = language_scope()
df.head(10)

,language,year,month,pushes,pull_requests,issues,issue_comments,stars,forks,creates,contributors,active_repos,date
0,JavaScript,2016,1,13650,9407,4768,11779,10509,2873,3146,18657,7393,2016-01-01
1,Java,2016,1,8235,4285,1871,5653,2728,1360,1732,7887,3797,2016-01-01
2,Python,2016,1,7311,4562,2070,7016,2733,1157,1327,8162,4153,2016-01-01
3,C#,2016,1,5262,1247,965,2638,601,383,346,2362,1094,2016-01-01
4,Ruby,2016,1,5149,4341,1200,3500,1360,1030,1574,4510,2618,2016-01-01
5,PHP,2016,1,4848,2919,1494,3987,1336,677,1034,4542,2593,2016-01-01
6,HTML,2016,1,4829,2932,815,2038,971,800,907,3786,1757,2016-01-01
7,C++,2016,1,3735,1918,1137,4383,1576,592,549,4890,1905,2016-01-01
8,CSS,2016,1,2642,1704,338,819,826,392,511,2197,965,2016-01-01
9,Shell,2016,1,2499,917,354,1352,857,338,259,2122,944,2016-01-01


## Basic shape and invariants

These checks use raw counts because they validate the extract itself: schema, missingness, duplicates, and impossible negative values.

In [38]:
summary = pd.Series(
    {
        "rows": len(df),
        "columns": df.shape[1],
        "languages": df["language"].nunique(),
        "months": df[["year", "month"]].drop_duplicates().shape[0],
        "min_date": df["date"].min().date(),
        "max_date": df["date"].max().date(),
        "duplicate_language_month_rows": df.duplicated(["language", "year", "month"]).sum(),
        "total_missing_values": df.isna().sum().sum(),
        "language_scope": "all GitHub labels" if INCLUDE_ALL_GITHUB_LANGUAGE_LABELS else "programming-language focused",
        "languages_in_scope": analysis_df["language"].nunique(),
    }
)
summary.to_frame("value")

,value
rows,34197
columns,13
languages,523
months,114
min_date,2016-01-01
max_date,2025-06-01
duplicate_language_month_rows,0
total_missing_values,0
language_scope,all GitHub labels
languages_in_scope,523


In [39]:
df.dtypes.to_frame("dtype")

,dtype
language,str
year,int64
month,int64
pushes,int64
pull_requests,int64
issues,int64
issue_comments,int64
stars,int64
forks,int64
creates,int64


In [40]:
df.isna().sum().to_frame("missing_values")

,missing_values
language,0
year,0
month,0
pushes,0
pull_requests,0
issues,0
issue_comments,0
stars,0
forks,0
creates,0


In [41]:
negative_counts = (df[metric_cols] < 0).sum().sort_values(ascending=False)
negative_counts.to_frame("negative_values")

,negative_values
pushes,0
pull_requests,0
issues,0
issue_comments,0
stars,0
forks,0
creates,0
contributors,0
active_repos,0


In [42]:
assert df.duplicated(["language", "year", "month"]).sum() == 0
assert df.isna().sum().sum() == 0
assert (df[metric_cols] >= 0).all().all()

## Normalized monthly shares

The CSV is based on the 15th day of each month, not full-month totals. Because weekday/weekend effects can move raw counts around, trend and ranking charts below normalize each metric within the month.

In [43]:
normalized = df.sort_values(["language", "date"]).copy()
for metric in metric_cols:
    monthly_total = normalized.groupby("date")[metric].transform("sum")
    normalized[f"{metric}_share"] = normalized[metric] / monthly_total.where(monthly_total != 0, np.nan)
    normalized[f"{metric}_share_pct"] = normalized[f"{metric}_share"] * 100
    normalized[f"{metric}_rank"] = normalized.groupby("date")[f"{metric}_share"].rank(
        method="first",
        ascending=False,
    )

share_cols = [f"{metric}_share" for metric in metric_cols]
share_check = normalized.groupby("date")[share_cols].sum().sub(1).abs().max().sort_values(ascending=False)
share_check.to_frame("max_abs_error_from_1")

,max_abs_error_from_1
pushes_share,1.110223e-16
pull_requests_share,1.110223e-16
forks_share,1.110223e-16
creates_share,1.110223e-16
issues_share,0.000000e+00
issue_comments_share,0.000000e+00
stars_share,0.000000e+00
contributors_share,0.000000e+00
active_repos_share,0.000000e+00


In [44]:
assert share_check.max() < 1e-9

In [45]:
language_summary = language_popularity_table(analysis_df)

# Higher score means the language has a larger average share across active repos,
# contributors, pull requests, pushes, and stars.
top_40_languages = language_summary.head(40).index.tolist()
top_20_languages = language_summary.head(20).index.tolist()
language_summary.head(40)

,active_repos,contributors,pull_requests,pushes,stars,popularity_score
language,,,,,,
JavaScript,2402936,3220189,3790494,4684392,1347238,0.177228
Python,1564732,2856341,2092623,3540629,1355453,0.134958
TypeScript,1498674,2016113,2879029,3754499,788898,0.120438
Java,1053077,1577953,1710443,2204577,503523,0.078705
Go,616600,1141173,1022173,1131529,612499,0.055498
HTML,605081,892861,1248200,1905993,179162,0.049489
C++,528827,1167182,614654,1028514,458581,0.045532
C#,425384,696386,679333,1334888,236178,0.036250
PHP,450640,612552,595931,882263,207822,0.031031


## Coverage checks

Coverage shows which language labels appear in each sampled month. This is an availability check, not a magnitude comparison.

In [46]:
rows_per_language = df.groupby("language").size().sort_values(ascending=False)
rows_per_language.describe().to_frame("rows_per_language")

,rows_per_language
count,523.000000
mean,65.386233
std,42.855768
min,1.000000
25%,21.000000
50%,72.000000
75%,112.500000
max,114.000000


In [47]:
coverage = (
    analysis_df.loc[analysis_df["language"].isin(top_40_languages)]
    .assign(present=1)
    .pivot_table(index="language", columns="date", values="present", fill_value=0)
    .reindex(top_40_languages)
)

fig = px.imshow(
    coverage,
    x=[date.strftime("%Y-%m") for date in coverage.columns],
    y=coverage.index,
    color_continuous_scale=[[0, "#f3f4f6"], [1, "#16a34a"]],
    zmin=0,
    zmax=1,
    aspect="auto",
    labels={"x": "Month", "y": "Language", "color": "Present"},
    title="Monthly coverage for top 40 languages by total active repos",
)
fig.update_layout(height=760, coloraxis_showscale=False)
fig.update_traces(hovertemplate="Language: %{y}<br>Month: %{x}<br>Present: %{z}<extra></extra>")
fig.show()

## Interactive normalized trend checks

These line charts show each language's share of the monthly total. Raw sampled counts are included in hover labels for context only.

In [48]:
def ordered_language_plot_data(data, languages):
    ordered = data.copy()
    ordered["language"] = pd.Categorical(
        ordered["language"],
        categories=languages,
        ordered=True,
    )
    return ordered.sort_values(["language", "date"])


def metric_share_trend(metric="active_repos", top_n=20, include_all=INCLUDE_ALL_GITHUB_LANGUAGE_LABELS):
    languages = top_languages_by_popularity(top_n=top_n, include_all=include_all)
    plot_df = normalized.loc[normalized["language"].isin(languages)].copy()
    if not include_all:
        plot_df = plot_df.loc[~plot_df["language"].isin(NON_GENERAL_PURPOSE_LABELS)]
    plot_df = ordered_language_plot_data(plot_df, languages)

    fig = px.line(
        plot_df,
        x="date",
        y=f"{metric}_share_pct",
        color="language",
        color_discrete_map=language_color_map(languages),
        category_orders={"language": languages},
        labels={
            "date": "Month",
            f"{metric}_share_pct": f"{metric.replace('_', ' ').title()} share (%)",
            "language": "Language",
        },
        title=f"Top {top_n} languages by composite popularity: {metric.replace('_', ' ')} monthly share",
        custom_data=["language", metric, f"{metric}_rank"],
    )
    fig.update_traces(
        mode="lines",
        hovertemplate=(
            "Language: %{customdata[0]}<br>"
            "Month: %{x|%Y-%m}<br>"
            "Share: %{y:.2f}%<br>"
            "Sampled count: %{customdata[1]:,}<br>"
            "Monthly rank: %{customdata[2]:.0f}<extra></extra>"
        ),
    )
    fig.update_layout(
        height=650,
        hovermode="x unified",
        legend_title_text="Language (most to least popular)",
        legend_traceorder="normal",
    )
    return fig

metric_share_trend(metric="active_repos", top_n=20).show()

In [49]:
metric_share_trend(metric="contributors", top_n=20).show()

## Latest month ranking

The bar chart ranks languages by normalized active repository share for the latest available month. Raw sampled values remain available in the hover tooltip.

In [50]:
latest_month = normalized["date"].max()
latest_metric = "active_repos"
latest_top = (
    normalized.loc[normalized["date"].eq(latest_month)]
    .pipe(lambda data: data if INCLUDE_ALL_GITHUB_LANGUAGE_LABELS else data.loc[~data["language"].isin(NON_GENERAL_PURPOSE_LABELS)])
    .sort_values(f"{latest_metric}_share", ascending=False)
    .head(40)
    .sort_values(f"{latest_metric}_share", ascending=True)
)

fig = px.bar(
    latest_top,
    x=f"{latest_metric}_share_pct",
    y="language",
    orientation="h",
    labels={
        f"{latest_metric}_share_pct": "Active repos share (%)",
        "language": "Language",
    },
    title=f"Top 40 languages by active repo share in {latest_month:%Y-%m}",
    custom_data=["active_repos", "contributors", "pushes", "pull_requests", "stars"],
)
fig.update_traces(
    marker_color="#2563eb",
    hovertemplate=(
        "Language: %{y}<br>"
        "Active repo share: %{x:.2f}%<br>"
        "Sampled active repos: %{customdata[0]:,}<br>"
        "Contributors: %{customdata[1]:,}<br>"
        "Pushes: %{customdata[2]:,}<br>"
        "Pull requests: %{customdata[3]:,}<br>"
        "Stars: %{customdata[4]:,}<extra></extra>"
    ),
)
fig.update_layout(height=820)
fig.show()

latest_top.sort_values(f"{latest_metric}_share", ascending=False)[
    ["language", f"{latest_metric}_share_pct", "active_repos", "contributors", "pushes", "pull_requests", "stars"]
].head(20)

,language,active_repos_share_pct,active_repos,contributors,pushes,pull_requests,stars
33892,TypeScript,18.489692,18197,20273,51150,35666,6954
33893,Python,15.432293,15188,23131,37935,27011,11666
33894,JavaScript,12.719347,12518,14922,31410,22760,4597
33896,Java,6.470427,6368,7959,14553,11563,1828
33897,Go,5.218611,5136,6666,10872,6785,3882
33895,HTML,4.463660,4393,5461,19063,19816,1177
33899,C++,4.257395,4190,6715,7722,4791,2679
33898,C#,3.705661,3647,4416,9343,6037,1198
33900,Rust,3.437414,3383,5540,6955,4492,3151
33905,C,2.767815,2724,3939,3875,2962,1711


## Rank over time

Rank movement is another way to compare languages without relying on raw sampled totals. Rank 1 is the largest monthly share.

In [51]:
def rank_over_time(metric="active_repos", top_n=20, include_all=INCLUDE_ALL_GITHUB_LANGUAGE_LABELS):
    rank_languages = top_languages_by_popularity(top_n=top_n, include_all=include_all)
    rank_df = normalized.loc[normalized["language"].isin(rank_languages)].copy()
    if not include_all:
        rank_df = rank_df.loc[~rank_df["language"].isin(NON_GENERAL_PURPOSE_LABELS)]
    rank_df = ordered_language_plot_data(rank_df, rank_languages)

    fig = px.line(
        rank_df,
        x="date",
        y=f"{metric}_rank",
        color="language",
        color_discrete_map=language_color_map(rank_languages),
        category_orders={"language": rank_languages},
        labels={
            "date": "Month",
            f"{metric}_rank": "Monthly rank",
            "language": "Language",
        },
        title=f"Top {top_n} languages by composite popularity: {metric.replace('_', ' ')} share rank over time",
        custom_data=["language", f"{metric}_share_pct", metric],
    )
    fig.update_yaxes(autorange="reversed", dtick=2)
    fig.update_traces(
        hovertemplate=(
            "Language: %{customdata[0]}<br>"
            "Month: %{x|%Y-%m}<br>"
            "Rank: %{y:.0f}<br>"
            "Share: %{customdata[1]:.2f}%<br>"
            "Sampled count: %{customdata[2]:,}<extra></extra>"
        )
    )
    fig.update_layout(
        height=650,
        hovermode="x unified",
        legend_title_text="Language (most to least popular)",
        legend_traceorder="normal",
    )
    return fig

rank_over_time(metric="active_repos", top_n=20).show()

### Contributor rank over time

This view ranks the same popularity-selected languages by their monthly contributor share.


In [52]:
rank_over_time(metric="contributors", top_n=20).show()

## Event mix over time

This chart compares the composition of sampled GitHub events each month. It answers which event types make up the activity mix, not whether total activity rose or fell.

In [53]:
monthly_event_mix = df.groupby("date", as_index=False)[event_metric_cols].sum()
monthly_event_mix_long = monthly_event_mix.melt(
    id_vars="date",
    value_vars=event_metric_cols,
    var_name="event_type",
    value_name="sampled_count",
)
monthly_event_mix_long["event_total"] = monthly_event_mix_long.groupby("date")["sampled_count"].transform("sum")
monthly_event_mix_long["event_share_pct"] = (
    monthly_event_mix_long["sampled_count"] / monthly_event_mix_long["event_total"] * 100
)
monthly_event_mix_long["event_type"] = monthly_event_mix_long["event_type"].str.replace("_", " ").str.title()

fig = px.area(
    monthly_event_mix_long,
    x="date",
    y="event_share_pct",
    color="event_type",
    labels={
        "date": "Month",
        "event_share_pct": "Share of sampled events (%)",
        "event_type": "Event type",
    },
    title="Monthly sampled event mix by share",
    custom_data=["event_type", "sampled_count"],
)
fig.update_traces(
    hovertemplate=(
        "Event: %{customdata[0]}<br>"
        "Month: %{x|%Y-%m}<br>"
        "Share: %{y:.2f}%<br>"
        "Sampled count: %{customdata[1]:,}<extra></extra>"
    )
)
fig.update_layout(height=620, hovermode="x unified")
fig.show()

## Outlier checks on normalized shares

Large month-over-month jumps in share can be real, but these tables are a useful place to look for suspicious spikes or language-specific changes.

In [54]:
mom = normalized.sort_values(["language", "date"]).copy()
for metric in ["active_repos", "contributors", "stars", "pushes"]:
    share_col = f"{metric}_share"
    prev_col = f"{metric}_share_prev"
    change_col = f"{metric}_share_pct_change"
    mom[prev_col] = mom.groupby("language")[share_col].shift(1)
    mom[change_col] = (mom[share_col] - mom[prev_col]) / mom[prev_col].replace(0, np.nan)

mom.loc[mom["language"].isin(top_40_languages)].sort_values(
    "active_repos_share_pct_change",
    ascending=False,
)[
    [
        "language",
        "date",
        "active_repos_share_prev",
        "active_repos_share",
        "active_repos_share_pct_change",
        "active_repos",
    ]
].head(25)

,language,date,active_repos_share_prev,active_repos_share,active_repos_share_pct_change,active_repos
20345,Groovy,2022-02-01,0.000253,0.002913,10.515531,500
13341,Scala,2020-04-01,0.001579,0.013167,7.338578,1683
20012,Vue,2022-01-01,0.011748,0.095173,7.101197,30846
13350,HCL,2020-04-01,0.000455,0.003231,6.105022,413
13353,Groovy,2020-04-01,0.000341,0.002026,4.940922,259
21986,Svelte,2022-07-01,0.001781,0.010325,4.797206,1838
3630,Svelte,2017-04-01,0.000035,0.000177,4.095190,6
13344,R,2020-04-01,0.001263,0.006126,3.849307,783
13359,Perl,2020-04-01,0.000445,0.002120,3.761327,271
20340,HCL,2022-02-01,0.000873,0.003980,3.557871,683


In [55]:
distribution_metric = "active_repos"
dist_df = normalized.loc[normalized[f"{distribution_metric}_share"] > 0].copy()
fig = px.histogram(
    dist_df,
    x=f"{distribution_metric}_share_pct",
    nbins=80,
    log_y=True,
    labels={f"{distribution_metric}_share_pct": "Active repos share (%)", "count": "Language-month rows"},
    title="Distribution of active repo monthly shares across language-month rows",
)
fig.update_traces(marker_color="#0f766e", hovertemplate="Share: %{x:.4f}%<br>Rows: %{y:,}<extra></extra>")
fig.update_layout(height=520)
fig.show()